In [31]:
import os
from dotenv import load_dotenv

load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")

print("OLLAMA API KEY FOUND:", bool(ollama_api_key))

OLLAMA API KEY FOUND: True


In [32]:
# !pip install -q langchain-ollama langchain-core requests python-dotenv

In [33]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests


**Creating Tool**

In [34]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [35]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [36]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


**tool binding**

In [37]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")

llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
    temperature=0.2,
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {ollama_api_key}"
        }
    }
)

In [38]:
# Send a simple message to the Ollama Cloud LLM
response = llm.invoke("Hi, How are you")

# Print only the text generated by the LLM
print(response.content)

Hello! I’m doing great, thanks for asking. How can I help you today?


In [39]:
# Bind the multiply tool to the LLM so it can call the tool when required
llm_with_tools = llm.bind_tools([multiply])

**Tool Calling**

In [51]:
# Send a normal message to the LLM with the multiply tool available
response = llm_with_tools.invoke('can you multiply 3 with 10')

print(response)

content='' additional_kwargs={} response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-09-21T09:48:15.271658112Z', 'done': True, 'done_reason': 'stop', 'total_duration': 516961321, 'load_duration': None, 'prompt_eval_count': 139, 'prompt_eval_duration': None, 'eval_count': 48, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'} id='lc_run--01a0c35d-b8b1-7082-b2ee-ff359f2a0d4b-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'e1c257e6-d7cc-4034-b42d-13ca23b1dcd6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 139, 'output_tokens': 48, 'total_tokens': 187}


In [52]:
# Send the query to the tool-enabled LLM and get the first tool call
result = llm_with_tools.invoke(
    "Can you multiply 3 with 10?"
)

In [53]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'd5c0419f-4bb7-474f-8839-5d3c84714660',
 'type': 'tool_call'}

**Tool Execution**

In [57]:
final_result = multiply.invoke(result.tool_calls[0])

In [58]:
# or 

# final_result = multiply.invoke({'name': 'multiply',
#  'args': {'a': 3, 'b': 10},
#  'id': 'acd110c6-db37-4f54-aa26-8d57285e37f5',
#  'type': 'tool_call'})

In [59]:
print(final_result)

content='30' name='multiply' tool_call_id='d5c0419f-4bb7-474f-8839-5d3c84714660'


In [ ]:
## ab en sabko llm ko phir se bhejna hota hai ,taki llm final outpur de sahi se  ,, esko next file me cover kiya hu  file ka name hai ,tool calling